# Notebook 4 — Supplier EDA

**Goal:** Profile supplier performance using purchase order data.

Key metrics:
- **Lead time**: actual vs. requested — how long does it really take?
- **OTIF** (On Time In Full): delivered on time AND complete quantity
- **Fill rate**: fraction of ordered quantity actually received
- **Late rate**: fraction of POs that arrived after requested delivery

This is the foundation for Project 6 (Supplier Risk Engine).

> **Interview question:** "How do you measure supplier performance?"
> Expected answer: Lead time distribution, OTIF, fill rate, cancellation rate, price variance.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sqlalchemy import create_engine

DB_URL = os.getenv('SUPPLY_CHAIN_DB_URL', '')
engine = create_engine(DB_URL)
SCHEMA = 'supply_chain'

def q(sql): return pd.read_sql(sql, engine)

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

## 1. Supplier performance summary (using analytical view)

In [ ]:
perf = q(f'SELECT * FROM {SCHEMA}.v_supplier_performance ORDER BY otif_rate DESC LIMIT 20')
print(perf[['supplier_name','supplier_type','total_pos','avg_actual_lead_days',
            'avg_late_days','late_rate','otif_rate','total_spend']].to_string(index=False))

## 2. Lead time comparison: National vs. Regional vs. Local

National suppliers should have shorter, less variable lead times. If this is not the case in our data, the simulation needs adjustment.

In [ ]:
lt_by_type = q(f"""
SELECT
    s.supplier_type,
    ROUND(AVG(po.actual_delivery - po.order_date), 2)            AS avg_actual_lt,
    ROUND(STDDEV(po.actual_delivery - po.order_date), 2)         AS std_actual_lt,
    ROUND(PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY po.actual_delivery - po.order_date), 1) AS median_lt,
    ROUND(PERCENTILE_CONT(0.90) WITHIN GROUP (ORDER BY po.actual_delivery - po.order_date), 1) AS p90_lt,
    ROUND(AVG(CASE WHEN po.is_late THEN 1.0 ELSE 0.0 END), 3)   AS late_rate,
    COUNT(*) AS po_count
FROM {SCHEMA}.purchase_orders po
JOIN {SCHEMA}.suppliers s ON s.supplier_id = po.supplier_id
WHERE po.status = 'Received' AND po.actual_delivery IS NOT NULL
GROUP BY s.supplier_type
ORDER BY avg_actual_lt
""")
print(lt_by_type.to_string(index=False))

In [ ]:
# Lead time distribution as box plot
lt_raw = q(f"""
SELECT s.supplier_type, (po.actual_delivery - po.order_date) AS actual_lt
FROM {SCHEMA}.purchase_orders po
JOIN {SCHEMA}.suppliers s ON s.supplier_id = po.supplier_id
WHERE po.status = 'Received' AND po.actual_delivery IS NOT NULL
  AND (po.actual_delivery - po.order_date) BETWEEN 1 AND 60
""")

fig, ax = plt.subplots()
groups = [lt_raw[lt_raw['supplier_type'] == t]['actual_lt'].values
          for t in ['National', 'Regional', 'Local']]
ax.boxplot(groups, labels=['National', 'Regional', 'Local'], patch_artist=True)
ax.set_title('Lead Time Distribution by Supplier Type (days)')
ax.set_ylabel('Actual Lead Time (days)')
plt.tight_layout()
plt.show()

## 3. OTIF rate over time

OTIF (On Time In Full) is the gold standard supplier KPI. Walmart famously penalizes suppliers for OTIF failures. A supplier with OTIF < 85% is considered unreliable.

In [ ]:
otif_monthly = q(f"""
SELECT
    DATE_TRUNC('month', po.order_date::timestamp)::date AS month,
    s.supplier_type,
    ROUND(1 - AVG(CASE WHEN po.is_late THEN 1.0 ELSE 0.0 END), 3) AS otif_rate
FROM {SCHEMA}.purchase_orders po
JOIN {SCHEMA}.suppliers s ON s.supplier_id = po.supplier_id
WHERE po.status = 'Received'
GROUP BY month, s.supplier_type
ORDER BY month, s.supplier_type
""")
otif_monthly['month'] = pd.to_datetime(otif_monthly['month'])

fig, ax = plt.subplots()
for stype in ['National', 'Regional', 'Local']:
    sub = otif_monthly[otif_monthly['supplier_type'] == stype]
    ax.plot(sub['month'], sub['otif_rate'], label=stype, marker='o', markersize=3)
ax.axhline(0.85, color='red', linestyle='--', linewidth=1, label='85% threshold')
ax.set_title('OTIF Rate Over Time by Supplier Type')
ax.set_xlabel('Month')
ax.set_ylabel('OTIF Rate')
ax.legend()
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 4. Fill rate distribution

Fill rate = received_qty / ordered_qty per PO line. A fill rate of 0.80 means the supplier only shipped 80% of what was ordered — the remaining 20% is unmet demand.

In [ ]:
fill = q(f"""
SELECT s.supplier_type, pol.fill_rate
FROM {SCHEMA}.purchase_order_lines pol
JOIN {SCHEMA}.purchase_orders po ON po.po_id = pol.po_id
JOIN {SCHEMA}.suppliers s ON s.supplier_id = po.supplier_id
WHERE po.status = 'Received' AND pol.fill_rate IS NOT NULL
""")

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
for ax, stype in zip(axes, ['National', 'Regional', 'Local']):
    data = fill[fill['supplier_type'] == stype]['fill_rate']
    ax.hist(data, bins=30, color='steelblue', edgecolor='white')
    ax.axvline(data.mean(), color='red', linestyle='--', label=f'Mean: {data.mean():.3f}')
    ax.set_title(f'{stype} Suppliers')
    ax.set_xlabel('Fill Rate')
    ax.legend()
fig.suptitle('Fill Rate Distribution by Supplier Type')
plt.tight_layout()
plt.show()

## 5. Worst-performing suppliers

These are candidates for supplier risk intervention in Project 6.

In [ ]:
worst_sups = q(f"""
SELECT supplier_name, supplier_type, total_pos, avg_actual_lead_days,
       late_rate, otif_rate, total_spend
FROM {SCHEMA}.v_supplier_performance
WHERE total_pos >= 10
ORDER BY otif_rate ASC
LIMIT 15
""")
print('Worst 15 suppliers by OTIF:')
print(worst_sups.to_string(index=False))

## 6. Seasonal lead time inflation

The simulation inflates lead times +20% in Nov–Dec. Can we detect this in the data? This is the kind of pattern that would justify building a seasonal supplier risk model.

In [ ]:
seasonal_lt = q(f"""
SELECT
    EXTRACT(MONTH FROM po.order_date) AS month,
    s.supplier_type,
    ROUND(AVG(po.actual_delivery - po.order_date), 2) AS avg_actual_lt
FROM {SCHEMA}.purchase_orders po
JOIN {SCHEMA}.suppliers s ON s.supplier_id = po.supplier_id
WHERE po.status = 'Received' AND po.actual_delivery IS NOT NULL
GROUP BY month, s.supplier_type
ORDER BY month, s.supplier_type
""")

fig, ax = plt.subplots()
for stype in ['National', 'Regional', 'Local']:
    sub = seasonal_lt[seasonal_lt['supplier_type'] == stype]
    ax.plot(sub['month'], sub['avg_actual_lt'], label=stype, marker='o')
ax.set_title('Average Lead Time by Month and Supplier Type')
ax.set_xlabel('Month')
ax.set_ylabel('Avg Lead Time (days)')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(['J','F','M','A','M','J','J','A','S','O','N','D'])
ax.legend()
plt.tight_layout()
plt.show()